In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Encoder(nn.Module):
    """Maps a spatial image batch to a flat feature representation vector"""

    def __init__(self, latent_dim=128):
        super().__init__()
        self.conv = nn.Sequential(
            # Input: [B, 3, 64, 64]
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),  # -> [B, 32, 32, 32]
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # -> [B, 64, 16, 16]
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),  # -> [B, 128, 8, 8]
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                128, 256, kernel_size=4, stride=2, padding=1
            ),  # -> [B, 256, 4, 4]
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        # Flatten spatial feature map down to vector space
        self.fc = nn.Linear(256 * 4 * 4, latent_dim)

    def forward(self, x):
        features = self.conv(x)
        features = features.view(features.size(0), -1)  # Flatten
        return self.fc(features)


class Decoder(nn.Module):
    """Maps a flat latent vector back up to a spatial image reconstruction"""

    def __init__(self, latent_dim=128):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 256 * 4 * 4)

        self.deconv = nn.Sequential(
            # Input shape: [B, 256, 4, 4]
            nn.ConvTranspose2d(
                256, 128, kernel_size=4, stride=2, padding=1
            ),  # -> [B, 128, 8, 8]
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(
                128, 64, kernel_size=4, stride=2, padding=1
            ),  # -> [B, 64, 16, 16]
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(
                64, 32, kernel_size=4, stride=2, padding=1
            ),  # -> [B, 32, 32, 32]
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(
                32, 3, kernel_size=4, stride=2, padding=1
            ),  # -> [B, 3, 64, 64]
            nn.Sigmoid(),  # Constrain final pixel intensities between 0.0 and 1.0
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(x.size(0), 256, 4, 4)  # Reshape back to spatial tensor dimensions
        return self.deconv(x)


In [2]:
class Autoencoder(nn.Module):
    """Standard Deterministic Bottleneck Autoencoder"""

    def __init__(self, latent_dim=128):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def forward(self, x):
        # Deterministic projection to latent space vector
        latent_z = self.encoder(x)
        # Reconstruct spatial target frame from z
        reconstruction = self.decoder(latent_z)
        return reconstruction


In [3]:
class VariationalAutoencoder(nn.Module):
    """Probabilistic Variational Autoencoder (VAE)"""

    def __init__(self, latent_dim=128):
        super().__init__()
        # Share the standard structural feature extractor convolutional blocks
        self.shared_encoder = Encoder(latent_dim=512)

        # Split individual projection heads for the statistical parameters
        self.fc_mu = nn.Linear(512, latent_dim)
        self.fc_logvar = nn.Linear(512, latent_dim)

        # Instantiate structural reconstruction block
        self.decoder = Decoder(latent_dim)

    def reparameterize(self, mu, logvar):
        """The Reparameterization Trick: z = mu + epsilon * sigma"""
        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)  # Sample random isotropic Gaussian noise
        return mu + (epsilon * std)

    def forward(self, x):
        # Extract features through standard backbone
        hidden = self.shared_encoder(x)

        # Predict Gaussian distribution parameters
        mu = self.fc_mu(hidden)
        logvar = self.fc_logvar(hidden)

        # Sample standard latent vector variables via reparameterization trick
        z = self.reparameterize(mu, logvar)

        # Reconstruct image from the sampled vector space coordinates
        reconstruction = self.decoder(z)
        return reconstruction, mu, logvar

    @staticmethod
    def calculate_loss(reconstruction, target, mu, logvar, kl_weight=0.01):
        """Computes the total loss: Reconstruction Loss + Kullback-Leibler Divergence"""
        # 1. Structural pixel reconstruction discrepancy (MSE)
        recon_loss = F.mse_loss(reconstruction, target, reduction="mean")

        # 2. KL Divergence: Forces the latent distribution to match a standard normal prior N(0, I)
        kl_loss = -0.5 * torch.mean(
            torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
        )

        # Combine losses using a scaling factor for the regularizer
        total_loss = recon_loss + (kl_weight * kl_loss)
        return total_loss, recon_loss, kl_loss


In [4]:
if __name__ == "__main__":
    # Create artificial batch: [Batch=2, Channels=3, Height=64, Width=64]
    mock_images = torch.rand(2, 3, 64, 64)
    print(f"Original Input Shape:             {mock_images.shape}")

    # Test Component 1 & 2: Standalone Encoder and Decoder
    enc = Encoder(latent_dim=128)
    dec = Decoder(latent_dim=128)
    z_vector = enc(mock_images)
    recon_images = dec(z_vector)
    print(f"-> Latent Code Vector Space Shape: {z_vector.shape}")
    print(f"-> Independent Decoder Out Shape:  {recon_images.shape}\n")

    # Test Component 3: Complete Autoencoder
    ae = Autoencoder(latent_dim=128)
    ae_out = ae(mock_images)
    print(f"Standard Autoencoder Output Shape: {ae_out.shape}\n")

    # Test Component 4: Variational Autoencoder (VAE) Pipeline
    vae = VariationalAutoencoder(latent_dim=128)
    vae_recon, mu, logvar = vae(mock_images)
    print(f"VAE Output Reconstruction Shape:   {vae_recon.shape}")
    print(f"VAE Target Distribution Mean Vector Shape:   {mu.shape}")
    print(f"VAE Target Distribution Log Variance Shape:  {logvar.shape}")

    # Verify custom loss calculation
    total_loss, r_loss, k_loss = VariationalAutoencoder.calculate_loss(
        vae_recon, mock_images, mu, logvar
    )
    print(f"-> VAE Combined Backpropagation Loss Value: {total_loss.item():.4f}")


Original Input Shape:             torch.Size([2, 3, 64, 64])
-> Latent Code Vector Space Shape: torch.Size([2, 128])
-> Independent Decoder Out Shape:  torch.Size([2, 3, 64, 64])

Standard Autoencoder Output Shape: torch.Size([2, 3, 64, 64])

VAE Output Reconstruction Shape:   torch.Size([2, 3, 64, 64])
VAE Target Distribution Mean Vector Shape:   torch.Size([2, 128])
VAE Target Distribution Log Variance Shape:  torch.Size([2, 128])
-> VAE Combined Backpropagation Loss Value: 0.1526
